In [ ]:
# Google Colab setup: select House_Price_data.csv when prompted.
from google.colab import files
from pathlib import Path
import shutil

Path('data').mkdir(exist_ok=True)
uploaded = files.upload()
source_name = next(iter(uploaded))
if not source_name.lower().endswith('.csv'):
    raise ValueError('Please upload House_Price_data.csv, not a notebook or PDF.')
shutil.move(source_name, 'data/House_Price_data.csv')
print('Dataset ready at data/House_Price_data.csv')

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path('data/House_Price_data.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['date'])
print(f'{df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

In [ ]:
df.info()
display(df.isna().sum().rename('missing values').to_frame())
print('Exact duplicate rows:', df.duplicated().sum())
print('Zero or negative prices:', (df['price'] <= 0).sum())
display(df.describe(include='all').T)

In [ ]:
df['street'] = df['street'].astype('string').str.strip()
df['city'] = df['city'].astype('string').str.strip()
df['statezip'] = df['statezip'].astype('string').str.strip()
df['country'] = df['country'].astype('string').str.strip()
df['sale_year'] = df['date'].dt.year
df['sale_month'] = df['date'].dt.to_period('M').astype(str)
df['home_age_at_sale'] = df['date'].dt.year - df['yr_built']
df['has_basement'] = (df['sqft_basement'] > 0).astype(int)
df['was_renovated'] = (df['yr_renovated'] > 0).astype(int)
invalid_price = df['price'] <= 0
analysis = df.loc[~invalid_price].copy()
print(f'Rows retained for price analysis: {len(analysis):,} / {len(df):,}')
print(f'Non-positive prices excluded: {invalid_price.sum():,}')
analysis.to_csv('outputs/cleaned_house_prices.csv', index=False)
analysis.head()

In [ ]:
from math import exp, lgamma, log, sqrt, erfc

# Small standard-library helpers for the exact t and F reference distributions.
def _beta_fraction(a, b, x):
    qab, qap, qam = a+b, a+1, a-1
    c, d = 1.0, 1.0-qab*x/qap
    d = 1e-300 if abs(d) < 1e-300 else 1.0/d
    h = d
    for m in range(1, 201):
        m2 = 2*m
        aa = m*(b-m)*x/((qam+m2)*(a+m2))
        d = 1.0 + aa*d; d = 1e-300 if abs(d) < 1e-300 else d
        c = 1.0 + aa/c; c = 1e-300 if abs(c) < 1e-300 else c
        d = 1.0/d; h *= d*c
        aa = -(a+m)*(qab+m)*x/((a+m2)*(qap+m2))
        d = 1.0 + aa*d; d = 1e-300 if abs(d) < 1e-300 else d
        c = 1.0 + aa/c; c = 1e-300 if abs(c) < 1e-300 else c
        d = 1.0/d
        delta = d*c; h *= delta
        if abs(delta-1.0) < 3e-14:
            break
    return h

def _regularized_beta(x, a, b):
    if x <= 0: return 0.0
    if x >= 1: return 1.0
    bt = exp(lgamma(a+b)-lgamma(a)-lgamma(b)+a*log(x)+b*log1p_neg(x))
    if x < (a+1)/(a+b+2):
        return bt*_beta_fraction(a,b,x)/a
    return 1-bt*_beta_fraction(b,a,1-x)/b

def log1p_neg(x):
    # stable log(1-x) for x in [0,1)
    import math
    return math.log1p(-x)

def t_two_sided_p(t, df):
    return _regularized_beta(df/(df+t*t), df/2, .5)

def t_critical_975(df):
    lo, hi = 0.0, 10.0
    for _ in range(70):
        mid = (lo+hi)/2
        if t_two_sided_p(mid, df) > .05: lo = mid
        else: hi = mid
    return (lo+hi)/2

def f_right_tail_p(f_value, df_num, df_den):
    x = df_den/(df_den+df_num*f_value)
    return _regularized_beta(x, df_den/2, df_num/2)

# Welch t-test: basement vs. no basement
has_basement = analysis['sqft_basement'].gt(0)
basement = analysis.loc[has_basement, 'price']
no_basement = analysis.loc[~has_basement, 'price']
t_stat = (basement.mean()-no_basement.mean()) / np.sqrt(basement.var(ddof=1)/len(basement) + no_basement.var(ddof=1)/len(no_basement))
mean_diff = basement.mean() - no_basement.mean()
se_diff = np.sqrt(basement.var(ddof=1)/len(basement) + no_basement.var(ddof=1)/len(no_basement))
df_welch = (basement.var(ddof=1)/len(basement) + no_basement.var(ddof=1)/len(no_basement))**2 / (
    (basement.var(ddof=1)/len(basement))**2/(len(basement)-1) +
    (no_basement.var(ddof=1)/len(no_basement))**2/(len(no_basement)-1))
tcrit = t_critical_975(df_welch)
t_p = t_two_sided_p(abs(t_stat), df_welch)
ci_low, ci_high = mean_diff - tcrit*se_diff, mean_diff + tcrit*se_diff
pooled_sd = np.sqrt(((len(basement)-1)*basement.var(ddof=1) + (len(no_basement)-1)*no_basement.var(ddof=1)) /
                     (len(basement)+len(no_basement)-2))
cohens_d = mean_diff / pooled_sd

# Chi-square: basement status x above-median price
above_median = analysis['price'] > analysis['price'].median()
contingency = pd.crosstab(has_basement, above_median)
observed = contingency.to_numpy()
row_totals, col_totals = observed.sum(axis=1), observed.sum(axis=0)
expected = np.outer(row_totals, col_totals) / observed.sum()
chi2 = ((observed-expected)**2/expected).sum()
chi_dof = (observed.shape[0]-1)*(observed.shape[1]-1)
chi_p = erfc(sqrt(chi2/2)) if chi_dof == 1 else float('nan')
cramers_v = np.sqrt(chi2 / contingency.to_numpy().sum())

# One-way ANOVA across condition ratings
condition_groups = [g['price'].to_numpy() for _,g in analysis.groupby('condition') if len(g)>1]
f_stat = (sum(len(g)*(g.mean()-analysis['price'].mean())**2 for g in condition_groups)/(len(condition_groups)-1)) / (
    sum(((g-g.mean())**2).sum() for g in condition_groups)/(len(analysis)-len(condition_groups)))
anova_p = f_right_tail_p(f_stat, len(condition_groups)-1, len(analysis)-len(condition_groups))
grand_mean = analysis['price'].mean()
ss_between = sum(len(g)*(g.mean()-grand_mean)**2 for g in condition_groups)
eta_squared = ss_between / ((analysis['price']-grand_mean)**2).sum()

print('WELCH T-TEST: basement minus no-basement mean price')
print(f'Group sizes: {len(basement):,} vs {len(no_basement):,}')
print(f'Mean difference: ${mean_diff:,.0f}; 95% CI [${ci_low:,.0f}, ${ci_high:,.0f}]')
print(f't={t_stat:.3f}; df={df_welch:.1f}; p={t_p:.4g}; Cohen d={cohens_d:.3f}')
print('Decision:', 'Reject H0' if t_p < .05 else 'Fail to reject H0')
print('\nCHI-SQUARE: basement status x above-median price')
display(contingency.rename(index={False:'No basement',True:'Basement'}, columns={False:'At/below median',True:'Above median'}))
print(f'chi-square={chi2:.3f}; df={chi_dof}; p={chi_p:.4g}; Cramers V={cramers_v:.3f}; min expected={expected.min():.1f}')
print('Decision:', 'Reject H0' if chi_p < .05 else 'Fail to reject H0')
print('\nONE-WAY ANOVA: mean price by condition rating')
print(f'F={f_stat:.3f}; p={anova_p:.4g}; eta-squared={eta_squared:.3f}')
print('Decision:', 'Reject H0' if anova_p < .05 else 'Fail to reject H0')
print('ANOVA is an overall test; it does not identify which pairs differ.')

In [ ]:
print('Median price:', f'${analysis.price.median():,.0f}')
print('Mean price:', f'${analysis.price.mean():,.0f}')
display(analysis[['price','sqft_living','sqft_lot','bedrooms','bathrooms']].describe().T)
city_summary = (analysis.groupby('city').agg(sales=('price','size'),
                 median_price=('price','median'), median_living=('sqft_living','median'))
                .query('sales >= 20').sort_values('median_price', ascending=False))
display(city_summary.head(10).round(0))

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(analysis['price'], bins=55, color='#218C8D', log_scale=(True, False))
plt.xlabel('Sale price (USD, logarithmic scale)'); plt.ylabel('Homes')
plt.title('House sale prices are strongly right-skewed'); plt.tight_layout(); plt.show()

In [ ]:
scatter_data = analysis.loc[analysis['price'] <= analysis['price'].quantile(.99)]
plt.figure(figsize=(9, 5))
sns.scatterplot(data=scatter_data.sample(min(1800, len(scatter_data)), random_state=42),
                x='sqft_living', y='price', hue='waterfront', alpha=.55, s=28,
                palette={0:'#218C8D', 1:'#E2A83B'})
plt.xlabel('Living area (sq ft)'); plt.ylabel('Sale price (USD)')
plt.title('Living area and price (through 99th price percentile)')
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9, 6))
plot_city = city_summary.head(10).sort_values('median_price')
plt.barh(plot_city.index, plot_city['median_price'], color='#218C8D')
plt.xlabel('Median sale price (USD)'); plt.ylabel('City')
plt.title('Median sale price by city (at least 20 sales)')
plt.tight_layout(); plt.show()

In [ ]:
corr_cols = ['price','bedrooms','bathrooms','sqft_living','sqft_lot','floors',
             'view','condition','sqft_above','sqft_basement','yr_built','yr_renovated']
plt.figure(figsize=(9, 7))
sns.heatmap(analysis[corr_cols].corr(numeric_only=True), cmap='vlag', center=0,
            linewidths=.3, cbar_kws={'label':'Pearson correlation'})
plt.title('Numeric feature correlations'); plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=analysis, x='condition', y='price', color='#76B7B2', showfliers=False)
plt.xlabel('Condition rating'); plt.ylabel('Sale price (USD)')
plt.title('Price distributions by reported condition (outlier points hidden)')
plt.tight_layout(); plt.show()